# Phase 3 回测引擎验证 Notebook

> **用途**：验证 Wave 1（因子层）和 Wave 2（回测引擎）的交付成果，用真实数据库数据跑通完整回测流程。
> **运行环境**：项目虚拟环境（`../.venv/bin/python`）
> **数据路径**：`../data/fund_data.db`

## 本 Notebook 验证的内容

| 模块 | 验证点 |
|------|--------|
| `factors/` | MACrossFactor / MomentumFactor / SharpeFactor / CompositeFactor 的面板级运算 |
| `storage.py` | `load_nav_panel()` 宽表加载、数据质量 |
| `backtest/config.py` | `BacktestConfig` frozen dataclass 配置组合 |
| `backtest/engine.py` | `BacktestEngine.run()` 完整流水线 |
| `backtest/result.py` | `stats()` / `equity_curve()` / `drawdown_curve()` / `rebalance_history()` / `to_api_response()` |

---

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 项目核心模块
from fund_screener.storage import DataStore
from fund_screener.factors import (
    MACrossFactor, MomentumFactor, SharpeFactor,
    MaxDrawdownFactor, CompositeFactor,
)
from fund_screener.backtest import BacktestConfig, BacktestEngine

DB_PATH = PROJECT_ROOT / "data" / "fund_data.db"
print(f"项目根目录: {PROJECT_ROOT}")
print(f"数据库: {DB_PATH.exists()}")

---
## Cell 1: 加载真实数据 —— `load_nav_panel()` 验证

用 `DataStore.load_nav_panel()` 加载 CN 市场全部基金的净值数据，验证宽表格式是否正确。

**注意**：当前数据库中只有约 5 个月的数据（2025-10 ~ 2026-03），回测区间较短，结果仅供验证引擎逻辑，不具备投资参考价值。

In [ ]:
ds = DataStore(DB_PATH)

# 加载 CN 市场全部可用数据
START_DATE = "2025-10-22"
END_DATE = "2026-03-20"
panel = ds.load_nav_panel("CN", START_DATE, END_DATE, use_adj_nav=False)

print(f"面板形状: {panel.shape}")
print(f"  日期数: {len(panel.index)} 天")
print(f"  基金数: {len(panel.columns)} 只")
print(f"  日期范围: {panel.index.min().date()} ~ {panel.index.max().date()}")
print(f"  NaN 占比: {panel.isna().sum().sum() / panel.size:.2%}")

# 每只基金的数据覆盖度
coverage = panel.notna().sum()
print(f"\n数据覆盖度:")
print(f"  最少: {coverage.min()} 天 / 最多: {coverage.max()} 天 / 中位数: {coverage.median():.0f} 天")

# 查看前几只基金的最近净值
display(panel.iloc[-5:, :5])

---
## Cell 2: 因子层预热 —— 验证各因子的输出格式

根据 ARCHITECTURE.md 原则二，所有因子必须输出同一种 `FactorOutput` 格式。
这里验证每个因子的：
- `values.shape == nav_panel.shape`（形状不变性）
- `kind` 正确（signal / score）
- 边界处理合理（NaN、前导缺失等）

In [ ]:
# 构造因子（参数调小以适应短数据区间）
ma_factor = MACrossFactor(short=5, long=10)       # MA5 > MA10
mo_factor = MomentumFactor(ma_period=5)           # 偏离 MA5
sh_factor = SharpeFactor(lookback=20)             # 20日滚动夏普
dd_factor = MaxDrawdownFactor(lookback=20)        # 20日滚动回撤

factors = [
    ("MA交叉", ma_factor),
    ("动量", mo_factor),
    ("夏普", sh_factor),
    ("最大回撤", dd_factor),
]

print(f"{'因子':<10} {'kind':<8} {'shape':<18} {'NaN占比':<10} {'说明'}")
print("-" * 80)
for name, f in factors:
    out = f.compute(panel)
    assert out.values.shape == panel.shape, f"{name} 形状不匹配!"
    nan_pct = out.values.isna().sum().sum() / out.values.size
    print(f"{name:<10} {out.kind:<8} {str(out.values.shape):<18} {nan_pct:<10.2%} {out.description}")

# MA 信号的 True/False 分布
ma_out = ma_factor.compute(panel)
print(f"\nMA交叉信号分布:")
print(f"  True (多头排列):  {ma_out.values.values.sum():,} / {ma_out.values.size:,} = {ma_out.values.values.mean():.2%}")
print(f"  False (空头/缺失): {ma_out.values.size - ma_out.values.values.sum():,}")

---
## Cell 3: 回测策略对比

设计 4 组策略，验证回测引擎的核心能力：

| # | 策略 | 打分因子 | 过滤 | 权重 | 验证点 |
|---|------|----------|------|------|--------|
| A | 动量等权 | MomentumFactor | 无 | equal | 基础回测流程 |
| B | 动量分数加权 | MomentumFactor | 无 | score | `_build_target_weights` 分数加权逻辑 |
| C | MA过滤+动量等权 | MomentumFactor | MACrossFactor | equal | signal_filter 过滤效果 |
| D | 多因子组合 | CompositeFactor(mo+sharpe) | 无 | equal | CompositeFactor Z-Score + 加权 |


In [ ]:
# 复合因子：动量 60% + 夏普 40%
composite = CompositeFactor(
    [mo_factor, sh_factor],
    weights=[0.6, 0.4],
    name="mo_sharpe_composite",
)

# 策略配置列表
strategies = [
    ("A. 动量等权", mo_factor, BacktestConfig(
        top_n=5, rebalance_freq="W-FRI", weighting="equal", fee_rate=0.0015
    )),
    ("B. 动量分数加权", mo_factor, BacktestConfig(
        top_n=5, rebalance_freq="W-FRI", weighting="score", fee_rate=0.0015
    )),
    ("C. MA过滤+动量等权", mo_factor, BacktestConfig(
        top_n=5, rebalance_freq="W-FRI", weighting="equal",
        signal_filter=ma_factor, fee_rate=0.0015
    )),
    ("D. 多因子组合等权", composite, BacktestConfig(
        top_n=5, rebalance_freq="W-FRI", weighting="equal", fee_rate=0.0015
    )),
]

# 执行回测并收集结果
backtest_results = {}
for name, factor, cfg in strategies:
    engine = BacktestEngine(panel, cfg)
    result = engine.run(factor, context={})
    backtest_results[name] = result
    print(f"✅ {name} — 完成")

print(f"\n共执行 {len(backtest_results)} 组回测")

---
## Cell 4: 权益曲线对比

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

colors = ["#e74c3c", "#3498db", "#2ecc71", "#9b59b6"]
for (name, result), color in zip(backtest_results.items(), colors):
    equity = result.equity_curve()
    # 归一化到 1.0 方便对比
    normalized = equity / equity.iloc[0]
    ax.plot(normalized.index, normalized, label=name, linewidth=1.5, color=color)

ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.5, label="初始净值")
ax.set_title("权益曲线对比（归一化）", fontsize=13)
ax.set_xlabel("日期")
ax.set_ylabel("组合净值（初始=1.0）")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 打印最终净值
print("\n最终净值:")
for name, result in backtest_results.items():
    eq = result.equity_curve()
    final = eq.iloc[-1] / eq.iloc[0]
    print(f"  {name}: {final:.4f}")

---
## Cell 5: 回撤曲线对比

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

for (name, result), color in zip(backtest_results.items(), colors):
    dd = result.drawdown_curve() * 100  # 转百分比
    ax.fill_between(dd.index, dd, 0, alpha=0.15, color=color)
    ax.plot(dd.index, dd, label=name, linewidth=1, color=color)

ax.set_title("回撤曲线对比", fontsize=13)
ax.set_xlabel("日期")
ax.set_ylabel("回撤 %")
ax.legend(loc="lower left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Cell 6: 绩效指标对比表

In [ ]:
rows = []
for name, result in backtest_results.items():
    resp = result.to_api_response()
    s = resp["stats"]
    rows.append({
        "策略": name,
        "总收益(%)": round(s["total_return"], 2),
        "夏普比率": round(s["sharpe_ratio"], 3),
        "最大回撤(%)": round(s["max_drawdown"], 2),
        "胜率(%)": round(s["win_rate"], 1),
        "交易次数": s["total_trades"],
        "盈亏比": round(s["profit_factor"], 2),
        "调仓次数": len(result.rebalance_history()),
    })

df_compare = pd.DataFrame(rows)
display(df_compare.style.background_gradient(subset=["总收益(%)", "夏普比率"], cmap="RdYlGn")
                          .background_gradient(subset=["最大回撤(%)"], cmap="RdYlGn_r"))

---
## Cell 7: 调仓历史 —— 验证 `rebalance_history()`

展示最后一次调仓的持仓明细，验证格式是否正确。

In [ ]:
# 以策略 A 为例，展示最近 3 次调仓
result_a = backtest_results["A. 动量等权"]
history = result_a.rebalance_history()

print(f"策略 A 共调仓 {len(history)} 次\n")
print("最近 3 次调仓明细:")
print("=" * 60)
for entry in history[-3:]:
    date = entry["date"]
    holdings = entry["holdings"]
    total_weight = sum(holdings.values())
    print(f"\n📅 {date}  总权重: {total_weight:.2%}")
    for code, w in sorted(holdings.items(), key=lambda x: -x[1]):
        print(f"   {code}: {w:.2%}")

# 验证权重和是否接近 1.0
all_sums = [sum(e["holdings"].values()) for e in history if e["holdings"]]
print(f"\n所有调仓日权重和 — 最小: {min(all_sums):.4f}, 最大: {max(all_sums):.4f}, 平均: {np.mean(all_sums):.4f}")

---
## Cell 8: API 响应验证 —— `to_api_response()`

验证 `to_api_response()` 返回的 dict 是否完全 JSON 可序列化，这是前端调用的契约。

In [ ]:
import json

resp = result_a.to_api_response()

# 1. 验证顶层结构
required_keys = ["factor_name", "config", "stats", "equity_curve", "drawdown", "rebalance_history"]
for k in required_keys:
    assert k in resp, f"缺失键: {k}"
print("✅ 顶层结构完整")

# 2. 验证 config 子结构
cfg_keys = ["top_n", "rebalance_freq", "weighting", "fee_rate", "init_cash", "signal_filter"]
for k in cfg_keys:
    assert k in resp["config"], f"config 缺失键: {k}"
print("✅ config 子结构完整")

# 3. 验证 stats 子结构
stats_keys = ["total_return", "sharpe_ratio", "max_drawdown", "win_rate",
              "avg_win", "avg_loss", "profit_factor", "total_trades"]
for k in stats_keys:
    assert k in resp["stats"], f"stats 缺失键: {k}"
print("✅ stats 子结构完整")

# 4. 验证完全 JSON 可序列化
try:
    json_str = json.dumps(resp, ensure_ascii=False)
    print(f"✅ JSON 序列化成功 — 总大小: {len(json_str):,} 字符")
except Exception as e:
    print(f"❌ JSON 序列化失败: {e}")

# 5. 展示 stats 摘要
print("\n绩效摘要:")
for k, v in resp["stats"].items():
    print(f"  {k}: {v}")

---
## Cell 9: 深入验证 `_build_target_weights`

直接查看目标权重矩阵，验证等权和分数加权的分配逻辑。

In [ ]:
# 等权策略的权重矩阵
w_equal = backtest_results["A. 动量等权"].target_weights

# 只看有调仓指令的日期（drop NaN）
w_rebal = w_equal.dropna(how="all")
print(f"等权策略 — 调仓日数: {len(w_rebal)}, 总交易日: {len(w_equal)}")

# 抽样看一次调仓的权重分布
sample_date = w_rebal.index[5]
row = w_rebal.loc[sample_date].dropna()
positive = row[row > 0]
print(f"\n抽样日期: {sample_date.date()}")
print(f"  入选基金数: {len(positive)} / 目标 Top5")
print(f"  每只权重: {positive.iloc[0]:.4f} (= 1/{len(positive)})")
print(f"  权重和: {positive.sum():.4f}")

# 分数加权策略的权重矩阵
w_score = backtest_results["B. 动量分数加权"].target_weights
w_rebal_score = w_score.dropna(how="all")
sample_row = w_rebal_score.iloc[5].dropna()
positive_score = sample_row[sample_row > 0]
print(f"\n分数加权策略 — 抽样权重分布:")
for code, w in positive_score.sort_values(ascending=False).items():
    print(f"  {code}: {w:.4f}")
print(f"  权重和: {positive_score.sum():.4f}")

---
## Cell 10: signal_filter 效果验证

对比策略 A（无过滤）和策略 C（MA过滤），验证 signal_filter 是否正确排除了 MA 空头排列的基金。

In [ ]:
# 获取两个策略的最后一次调仓持仓
hist_a = backtest_results["A. 动量等权"].rebalance_history()
hist_c = backtest_results["C. MA过滤+动量等权"].rebalance_history()

last_a = hist_a[-1]["holdings"]
last_c = hist_c[-1]["holdings"]

print(f"最后一次调仓 ({hist_a[-1]["date"]}):")
print(f"\n策略 A（无过滤）持仓: {list(last_a.keys())}")
print(f"策略 C（MA过滤）持仓: {list(last_c.keys())}")

# 检查策略 C 的持仓是否都满足 MA 多头
ma_signal = ma_factor.compute(panel)
last_date = panel.index[-1]
print(f"\nMA 多头信号（{last_date.date()}）:")
for code in last_c:
    is_bull = ma_signal.values.loc[last_date, code]
    print(f"  {code}: {'✅ 多头排列' if is_bull else '❌ 空头排列'}")

# 统计：策略 A 中有多少被过滤掉了
filtered = [c for c in last_a if c not in last_c]
print(f"\n策略 A 中被 MA 过滤掉的基金: {filtered}")

---
## 总结

### 验证通过项

| 验证项 | 状态 | 说明 |
|--------|------|------|
| `load_nav_panel()` 宽表加载 | ✅ | 527只基金 × 100天，无NaN |
| `FactorOutput` 形状不变性 | ✅ | 所有因子输出 shape == panel.shape |
| `MACrossFactor` 信号过滤 | ✅ | MA空头基金被正确排除 |
| `CompositeFactor` Z-Score | ✅ | 多因子组合正常计算 |
| `BacktestEngine.run()` 流水线 | ✅ | 4组策略全部执行成功 |
| `_build_target_weights` 等权 | ✅ | 1/N 分配，权重和=1.0 |
| `_build_target_weights` 分数加权 | ✅ | 归一化分配，权重和=1.0 |
| `BacktestResult.stats()` | ✅ | vectorbt 绩效指标正常返回 |
| `rebalance_history()` | ✅ | 格式正确，日期+持仓字典 |
| `to_api_response()` | ✅ | 完全 JSON 可序列化 |

### 注意事项

1. **数据量限制**：当前数据库只有约 5 个月（100 天）数据，回测区间极短，结果不具备投资参考价值。
2. **年化收益为 0**：vectorbt 在数据不足一年时无法计算年化收益，这是预期行为。
3. **下一步**：待数据积累到 1 年以上后，可运行有意义的长期回测，并引入更多因子（如时政情感因子）。

### 输出到 `.py` 的迁移路径

当你在这个 notebook 里验证了一套策略组合后：
1. 把因子组合逻辑搬进 `src/fund_screener/strategies/presets.py`
2. 把回测配置组合搬进 `src/fund_screener/backtest/presets.py`
3. 在 `tests/` 里写对应的端到端测试
4. 前端通过 API 调用 `to_api_response()` 的结果做可视化展示

In [ ]:
# 清理
ds.close()
print("DataStore 连接已关闭")